# **Trabajo Práctico Final**
---
## ***Procesamiento del Lenguaje Natural - TUIA***

| Autora | Legajo |
| --- | --- |
Rizzotto, María Camila | R-4676/1

# Ejercicio 1 - RAG

## Carga del Repositorio Pradera

Para tener acceso a las 3 fuentes de datos del juego que me fue asignado

In [1]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.colab import auth
import io
import os

# Autenticar
auth.authenticate_user()
drive_service = build('drive', 'v3')

# ID de la carpeta en Drive con los datos del juego PRADERA
FOLDER_ID = '1bQazMZnk73rRaG5btORykstehZABNf-C'
DESTINO_LOCAL = '/content/datos_pradera'

# Crear carpeta local si no existe
os.makedirs(DESTINO_LOCAL, exist_ok=True)

# Función recursiva para descargar carpetas con subcarpetas
def descargar_carpeta(folder_id, ruta_destino):
    query = f"'{folder_id}' in parents and trashed=false"
    resultados = drive_service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = resultados.get('files', [])

    for item in items:
        nombre = item['name']
        file_id = item['id']
        tipo = item['mimeType']

        if tipo == 'application/vnd.google-apps.folder':
            # Es una subcarpeta
            nueva_ruta = os.path.join(ruta_destino, nombre)
            os.makedirs(nueva_ruta, exist_ok=True)
            descargar_carpeta(file_id, nueva_ruta)
        else:
            # Es un archivo, lo descargamos
            request = drive_service.files().get_media(fileId=file_id)
            fh = io.FileIO(os.path.join(ruta_destino, nombre), 'wb')
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()

# Ejecutar la descarga
descargar_carpeta(FOLDER_ID, DESTINO_LOCAL)
print(f"✅ Carpeta descargada con subcarpetas en: {DESTINO_LOCAL}")

✅ Carpeta descargada con subcarpetas en: /content/datos_pradera


##Base de Datos vectorial


Para este ejercicio, voy a utilizar Chroma con LangChain como base de datos vectorial. En cuanto al modelo de embedding, usaré 'distiluse-base-multilingual-cased-v1' ya que yo tengo textos en varios idiomas. Y para el split de texto, vuelvo a utilizar spaCy como en mi anterior trabajo, pero agrupando de a 3 oraciones

In [2]:
%%capture
!pip install sentence-transformers chromadb langchain spacy
!python -m spacy download xx_ent_wiki_sm
!pip install -U langchain-community langchain-chroma

In [3]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import spacy

### Text Split

In [4]:
# Cargamos modelo SpaCy multilingue para la segmentación
nlp = spacy.load("xx_ent_wiki_sm")
nlp.add_pipe("sentencizer") #detecta los límites de oración basado en signos de puntuación

# Leemos todos los archivos .txt de la carpeta 'informacion'
folder_path  = Path("/content/datos_pradera/informacion")
documentos = []
documentos_fragmentados = []
for archivo in folder_path.glob("*.txt"):
    with open(archivo, encoding="utf-8") as f:
        texto = f.read()
        doc = Document(page_content=texto, metadata={"fuente": archivo.name})
        documentos.append(doc)
        doc_spacy = nlp(texto)

        # Agrupamos cada 3 oraciones como un fragmento temático básico
        oraciones = list(doc_spacy.sents)
        for i in range(0, len(oraciones), 3):
            fragmento = " ".join([str(s) for s in oraciones[i:i+3]])
            if fragmento.strip():
                documentos_fragmentados.append(Document(page_content=fragmento, metadata={"fuente": archivo.name}))

print(f"Documentos originales: {len(documentos)}")
print(f"Fragmentos generados: {len(documentos_fragmentados)}")

Documentos originales: 15
Fragmentos generados: 1884


### Cargo modelo de embeddings

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/distiluse-base-multilingual-cased-v1"
)

/tmp/ipython-input-5-1452574728.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

2_Dense/pytorch_model.bin:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

### Base de Datos

In [7]:
from chromadb.config import Settings
from langchain_chroma import Chroma
import logging

# Silenciar error de telemetría
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

# Desactivar telemetría en la configuración
chroma_settings = Settings(anonymized_telemetry=False)

# Creamos base de datos vectorial
db = Chroma.from_documents(
    documents=documentos_fragmentados,
    embedding=embedding_model,
    persist_directory="./chroma_pradera",
    client_settings=chroma_settings
)

In [89]:
# Creamos base de datos vectorial
db = Chroma.from_documents(
    documents=documentos_fragmentados,
    embedding=embedding_model,
    persist_directory="./chroma_pradera"
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Interfaz

In [8]:
#Cuando encontramos una respuesta que no esté en español, la vamos a traducir
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.1 MB/s eta 0:00:00


In [9]:
from deep_translator import GoogleTranslator

def traducir_a_espanol(texto):
    try:
        return GoogleTranslator(source='auto', target='es').translate(texto)
    except Exception as e:
        print("Error al traducir:", e)
        return texto

In [10]:
from langchain.schema import Document

def busqueda_semantica(query, k=3):
    """Función que, a partir de una consulta en lenguaje natural, devuelve k fragmentos más relevantes buscados de forma semántica en
     la base de datos vectorial de Chroma (db). Devuelve objetos Document compatibles con el retriever."""
    resultados = db.similarity_search(query, k=k)
    documentos = [
        Document(
            page_content=r.page_content,
            metadata={"fuente": r.metadata["fuente"]}
        )
        for r in resultados
    ]
    return documentos

### Consultas

In [11]:
import os
os.environ["CHROMA_TELEMETRY"] = "FALSE" #Para que no salga el warning de chroma y ensucie las consultas

In [12]:
query = "Cual es la mecánica del juego?"
resultados = busqueda_semantica(query, k=4)

print("Query:", query, "\n")

for i, doc in enumerate(resultados, 1):
    traduccion = traducir_a_espanol(doc.page_content)
    fuente = doc.metadata.get("fuente", "Desconocida")

    print(f"Respuesta {i}:\n{traduccion}")
    print(f"- Fuente: {fuente}\n")

Query: Cual es la mecánica del juego? 

Respuesta 1:
Esto no quiere decir que ese motor de juego no sea notable; Meadow presenta algunas mecánicas muy inteligentes que equilibran el juego. Por ejemplo, las dos avenidas de puntuación, los cuadros y las bonificaciones del tablero de fogatas, están bien ponderadas entre sí dadas la frecuencia con la que se juega cada uno, y la puntuación generalmente no es demasiado oscuro. El uso de cuatro mazos separados, cada uno con su propia distribución de tipos de tarjetas y símbolos, para alimentar el grupo de tarjetas compartidas asegura una disponibilidad abundante de tipos de tarjetas y símbolos para todos los jugadores y evita que los jugadores sigan con éxito una estrategia molesta de recursos de asfixia.
- Fuente: foro_reviews.txt

Respuesta 2:
El juego en sí es bastante sencillo, ya que la mayoría de las veces, quieres jugar cartas que muestren símbolos que necesitarás para que jueguen otras cartas: puede parecer un poco abstracta, pero nue

In [50]:
query = "how do i win"
resultados = busqueda_semantica(query, k=4)

print("Query:", query, "\n")

for i, doc in enumerate(resultados, 1):
    traduccion = traducir_a_espanol(doc.page_content)
    fuente = doc.metadata.get("fuente", "Desconocida")

    print(f"Respuesta {i}:\n{traduccion}")
    print("   (Respuesta original:",doc.page_content,")") #evidencia lo que fue traducido
    print(f"- Fuente: {fuente}\n")

Query: how do i win 

Respuesta 1:
Puedes ganar Meadow si puedes mostrar los puntos más de victoria después de un número determinado de rondas. 

¿Cómo funciona? 

Antes del primer juego, debes armar algunos soportes para tarjetas de cartón.
   (Respuesta original: Gewinnen kann man Meadow, wenn man nach einer vorgegebenen Rundenanzahl die meisten Siegpunkte vorweisen kann. 

Wie läuft das ab? 

Vor dem ersten Spiel muss man ersteinmal ein paar Kartenhalter aus Pappe zusammenbauen. )
- Fuente: foro_reviews.txt

Respuesta 2:
Lo máximo que he logrado llegar hasta ahora es como 5-7. Era la única forma en que logré ganar ese juego porque la persona contra la que estaba jugando obtuve todos los puntos de bonificación, y no obtuve ninguno, ¡así que comencé a concentrarme en las carreteras y tuve mucha suerte con las cartas que aparecieron! 
Sí, las carreteras son una muy buena estrategia cuando los demás no lo hacen ;-)

Hola, tengo una tarjeta con un solo requisito.
   (Respuesta original: 

In [20]:
query = "jugadores maximos permitidos"
resultados = busqueda_semantica(query, k=4)

print("Query:", query, "\n")

for i, doc in enumerate(resultados, 1):
    traduccion = traducir_a_espanol(doc.page_content)
    fuente = doc.metadata.get("fuente", "Desconocida")

    print(f"Respuesta {i}:\n{traduccion}")
    print(f"- Fuente: {fuente}\n")

Query: jugadores maximos permitidos  

Respuesta 1: ¿No hay límites de mano con tarjetas? ¿No hay límites de tokens de carretera?: ¿No es este juego ni límite de tamaño de la mano ni límite de tokens de carretera por jugador?
- Fuente: foro_general.txt  

Respuesta 2: Fuera de mi cabeza, la única restricción que conozco (juego base) es que no puedes colocar nada encima de una carta con el ícono Ungulado (Cabeza de los ciervos). Por supuesto, no todos tendrán los ungulados en juego (ya que dependen de los sobres de apertura), por lo que incluso esa restricción puede no aplicarse ...

2 preguntas menores: una tarjeta de paisaje nunca puede tener más de otra tarjeta (descubrimiento) colocada encima, ¿verdad? 

¿Existe una razón específica por la cual las fichas de carretera deben colocarse bajo paisajes?
- Fuente: foro_reglas.txt  

Respuesta 3: A través del juego podrás obtener Max 24 (en el juego 1, 2, 3 jugadores) o 16 (en el juego de 4 jugadores) Tokens de carretera si usas todos tus 

##Acceso a los Datos Estadísticos

Carga de información en df de Pandas

In [175]:
import pandas as pd

#Importamos datos de estadísticas en un dataframe
estadisticas_pradera = pd.read_csv("/content/datos_pradera/estadisticas/meadow_stats.csv", sep=",")
estadisticas_pradera

,Estadistica,Valor
0,Avg. Rating,7.719
1,No. of Ratings,"12,201"
2,Std. Deviation,1.22
3,Weight,2.25 / 5
4,Comments,"1,861"
5,Fans,"1,164"
6,Page Views,"983,188"
7,Overall Rank,209
8,Strategy Rank,158
9,Family Rank,28


### Procesamiento de información

In [176]:
#Recopilado información de importancia
#drop filas irrelevantes (como aquellas estadísticas muy relativas a la página y no al juego)
estadisticas_relevantes_pradera = estadisticas_pradera.drop([2,4,6,11,13,14,15,17,18])

In [177]:
# Limpieza de datos
estadisticas_relevantes_pradera["Estadistica"] = estadisticas_relevantes_pradera["Estadistica"].str.strip()
estadisticas_relevantes_pradera["Valor"] = estadisticas_relevantes_pradera["Valor"].astype(str).str.strip()

# Creamos un diccionario resumen
estadisticas = dict(zip(estadisticas_relevantes_pradera["Estadistica"], estadisticas_relevantes_pradera["Valor"]))

# Construimos el string para el LLM
info_llm = "\n".join([f"{k}: {v}" for k, v in estadisticas.items()])
print(info_llm)

Avg. Rating: 7.719
No. of Ratings: 12,201
Weight: 2.25 / 5
Fans: 1,164
Overall Rank: 209
Strategy Rank: 158
Family Rank: 28
All Time Plays: 50,530
Own: 23,140
Wishlist: 4,854


In [171]:
#tomar como indice a la columa 'Estadística':
estadisticas_relevantes_pradera.set_index('Estadistica', inplace=True)

### Uso de un modelo de lenguaje

Como esta tarea es simple, voy a usar un modelo liviano: Mistral-7B-Instruct GGUF

In [16]:
!pip install llama-cpp-python --upgrade --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 MB 10.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00


In [140]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

#Descargar modelo
model_path = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q4_K_M.gguf"
)

#Cargarlo con Llama
llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [18]:
def extraer_filtro_estadistico(pregunta_usuario, info_llm):
    prompt = f"""<|system|>
Eres un modelo experto en datos estadísticos de juegos de mesa.
Tu tarea es responder preguntas sobre el juego devolviendo el **nombre exacto del campo** que contiene la respuesta.
Debes elegir solo uno, el más adecuado, **entre los campos listados en la información**.

Información disponible:
{info_llm}

Formato esperado:
'<nombre exacto del campo>'

Ejemplos:

Usuario: ¿Cuántas personas agregaron el juego a su wishlist?
Respuesta:
'Wishlist'

Usuario: ¿Cuál es el puntaje promedio del juego?
Respuesta:
'Avg. Rating'

Usuario: ¿Cuántos fans tiene el juego?
Respuesta:
'Fans'

Usuario: ¿Cuántas veces se jugó el juego en total?
Respuesta:
'All Time Plays'

Usuario: {pregunta_usuario}
Respuesta:
"""

    respuesta = llm(prompt, max_tokens=200, stop=["Usuario:"])
    return respuesta["choices"][0]["text"].strip()


In [58]:
respuesta = extraer_filtro_estadistico("¿Cuántas veces se jugó el juego en total?", info_llm)
print(respuesta)

'All Time Plays'


In [80]:
respuesta = extraer_filtro_estadistico("en que ranking esta el juego", info_llm)
print(respuesta)

'Overall Rank'


In [82]:
respuesta = extraer_filtro_estadistico("cuantos fans tiene meadow", info_llm)
print(respuesta)

'Fans'


### Interfaz

In [19]:
def filtrar_set_estadistico(filtro):
  """Esta funcion filtra el set de datos según el resultado del LLM usado y obtiene una respuesta"""
  filtro = filtro.replace("'", "").replace('"', '')
  valor_estadistico = estadisticas_relevantes_pradera[estadisticas_relevantes_pradera["Estadistica"] == filtro]
  return valor_estadistico

def busqueda_tabular(consulta):
  """Se integran todas funciones realizadas: pasamos una consulta en lenguaje natural, un LLM obtiene un filtro y se trae la respuesta del set de"""
  filtro = extraer_filtro_estadistico(consulta, info_llm)
  valor_estadistico = filtrar_set_estadistico(filtro)
  return valor_estadistico

In [96]:
busqueda_tabular("que ranking de estrategia ocupa el juego?")

,Estadistica,Valor
8,Strategy Rank,158


##Base de Datos de Grafos - Neo4j + Cypher

### Creación Base de Datos de Grafo

In [20]:
!pip install py2neo pandas sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.2/177.2 kB 12.7 MB/s eta 0:00:00


Importamos archivo CSV -> Dataframe Pandas

In [21]:
from py2neo import Graph, Node, Relationship
import pandas as pd
#Importamos datos de relaciones en un dataframe
relaciones_pradera = pd.read_csv("/content/datos_pradera/relaciones/relaciones_juego.csv", sep=",")

In [22]:
relaciones_pradera.head()

,SUJETO1,RELACION,SUJETO2
0,Meadow,NOMBRE_ALTERNATIVO,Łąka
1,Meadow,NOMBRE_ALTERNATIVO,Livada
2,Meadow,NOMBRE_ALTERNATIVO,Meadow Im Reich der Natur
3,Meadow,NOMBRE_ALTERNATIVO,Na louce
4,Meadow,NOMBRE_ALTERNATIVO,Pradera


Conexión a mi instancia de Neo4j

In [137]:
graph = Graph("neo4j+s://3ccc91f3.databases.neo4j.io", auth=("neo4j", "wWhDo2bDgYcsoaLV9ZueOLnqGDpdH9qCB4qM0WIaHKg"))
# Verificación de conexión
graph.run("RETURN 'Conexión exitosa con Neo4j!' AS mensaje").data()

[{'mensaje': 'Conexión exitosa con Neo4j!'}]

Inserción de datos (creación BBDD)

In [22]:
# Inserto datos en grafo
for _, row in relaciones_pradera.iterrows():
    s1 = str(row["SUJETO1"]).strip()
    rel = str(row["RELACION"]).strip()
    s2 = str(row["SUJETO2"]).strip()

    # Crear nodos si no existen
    nodo1 = Node("Entidad", nombre=s1)
    nodo2 = Node("Entidad", nombre=s2)
    graph.merge(nodo1, "Entidad", "nombre")
    graph.merge(nodo2, "Entidad", "nombre")

    # Crear relación
    relacion = Relationship(nodo1, rel.upper().replace(" ", "_"), nodo2)
    graph.merge(relacion)

###Implementación modelo de lenguaje: consulta lenguaje natural -> consulta Cypher

Voy a utilizar un modelo de Sentence Transformer para obtener rápida y fácilmente una interfaz de consulta traduciendo del lenguaje natural -> consultas cypher, mediante una base de preguntas y consultas

In [24]:
def traducir_a_cypher(pregunta):
    prompt = f"""<|system|>
Sos un experto en bases de datos de grafos del juego 'Meadow' (en ingles), o 'Pradera'(en español). Tu tarea es convertir preguntas en lenguaje natural sobre el juego Meadow en consultas Cypher válidas.

Solo podes utilizar las siguientes relaciones:
- NOMBRE_ALTERNATIVO
- ANIO_RELEASE
- DISEÑADOR
- ILUSTRADOR
- RELACION_INTERNA
- EDITORIAL
- DESARROLLADOR
- DISEÑADOR GRAFICO
- EDITOR
- ESCRITOR
- CATEGORIA
- MECANISMO
- FAMILIA

Reglas:
- Si no reconoces una relacion valida entre las listadas, responde exactamente con: 'NO_MATCH'
- NO inventes relaciones ni nombres de entidades
- EVITA filtros muy complejos en las consultas. Hacelas lo mas parecidas posible a los ejemplos a continuacion.

Ejemplos:
Usuario: cuantos nombres alternativos tiene el juego?
Cypher: 'MATCH  p=()-[:NOMBRE_ALTERNATIVO]->() RETURN COUNT(p)'

Usuario: cuales son los nombres alternativos del juego?
Cypher: 'MATCH (p:Entidad)-[:NOMBRE_ALTERNATIVO]->(n:Entidad) RETURN n.nombre'

Usuario: Que mecanismos tiene el juego?
Cypher: 'MATCH p=()-[:MECANISMO]->() RETURN p'

Usuario: ¿Quién ilustró Meadow?
Cypher: 'MATCH (p:Entidad)-[:ILUSTRADOR]->(n:Entidad) RETURN n.nombre'

Usuario: en que año se lanzo Meadow?
Cypher: 'MATCH p=()-[:ANIO_RELEASE]->() RETURN p'

Usuario: cuales personas trabajaron juntas?
Cypher: ''MATCH (p:Entidad)-[:RELACION_INTERNA]->(q:Entidad) RETURN p.nombre, q.nombre''

Usuario: Que premios ganó el juego?
Cypher: 'NO_MATCH'

Usuario: cuantos jugadores pueden haber?
Cypher: 'NO_MATCH'

Usuario: {pregunta}
Cypher:"""

    respuesta = llm(prompt, max_tokens=200, stop=["Usuario:"])
    return respuesta["choices"][0]["text"].strip()

Ejemplo de uso y consultas

In [21]:
#Query
pregunta = "idiomas en que viene el juego"

#Implemento interfaz
resultado = traducir_a_cypher(pregunta)

#Imprimo
print(resultado)

'NO_MATCH'


In [19]:
#Query
pregunta = "en que año se lanzo el juego"

#Implemento interfaz
resultado = traducir_a_cypher(pregunta)

#Imprimo
print(resultado)

'MATCH p=()-[:ANIO_RELEASE]->() RETURN p'


In [41]:
#Query
pregunta = "cuantos nombre alternativos tiene el juego"

#Implemento interfaz
resultado = traducir_a_cypher(pregunta)

#Imprimo
print(resultado)

'MATCH p=()-[:NOMBRE_ALTERNATIVO]->() RETURN count(p)'


In [25]:
#Función que sistematiza todo el proceso
def busqueda_grafo(consulta):
    """Esta función recibe una consulta y devuelve la información brindada directamente por la base de datos de grafos"""
    #Obtener consulta Cypher generada por el LLM
    cypher = traducir_a_cypher(consulta)
    # print("Consulta Cypher generada:", cypher)
    cypher = cypher.replace("'", "").replace('"', '')

    #Verificar si la respuesta es inválida
    if cypher.strip() == "NO_MATCH":
        return "No se pudo encontrar una relación válida para esta pregunta en la base de datos de grafo."

    #Ejecutar la consulta Cypher
    try:
        resultado = graph.run(cypher).data()
        return resultado
    except Exception as e:
        print("Error al ejecutar la consulta:", e)
        return None

In [64]:
busqueda_grafo("de a cuantas personas se puede jugar meadow")

'No se pudo encontrar una relación válida para esta pregunta en la base de datos de grafo.'

In [65]:
busqueda_grafo("cuantos nombres alternativos tiene el juego")

[{'cantidad': 11}]

## Clasificador de Intención

En este punto voy a comparar modelos que reciben una consulta y la categoricen entre la fuente de datos que pueda llegar a responder esa pregunta entre estadísticas,
información y relaciones.
Por ejemplo:
- ¿Cómo gano en el ajedrez? -> Información
- ¿Quién trabajó para el ta-te-ti? -> Relaciones
- ¿Qué puntaje tienen las damas? -> Estadística

###Modelo del TP anterior

Había realizado 2: uno con regresión logística y otro de redes neuronales. El último fue el que mejores resultados me dio, así que lo voy a traer a este colab:

In [116]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
from sentence_transformers import SentenceTransformer

#Cargo modelo guardado llamado modelo_NN.keras
modelo_NN = tf.keras.models.load_model('modelo_NN.keras')

#Modelo de embedding que había usado
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

#Predicciones
def predecir_categoria_nn(frases, mostrar_resultados=True):
    frases = [f.lower() for f in frases]
    embeddings = model.encode(frases)
    predicciones = modelo_NN.predict(embeddings)
    etiquetas = np.argmax(predicciones, axis=1)
    mapeo_inverso = {0: "estadísticas", 1: "información", 2: "relaciones"}
    categorias_predichas = [mapeo_inverso[p] for p in etiquetas]

    #Mostrar resultados
    if mostrar_resultados:
      for frase, categoria in zip(frases, categorias_predichas):
          print(f"'{frase}' → {categoria}")
    return categorias_predichas

In [98]:
#Probamos algunas predicciones
predecir_categoria_nn(["quien creo el juego","cuantos usuarios juegan","como desempato"])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
'quien creo el juego' → relaciones
'cuantos usuarios juegan' → estadísticas
'como desempato' → relaciones


Acá se nota que se confundió en la última respuesta que debía ser 'información' y no 'relaciones'. Veremos si ahora un modelo con LLM tiene mejor performance

### Basado en LLM con Few-Shot Prompting

Voy a reciclar el modelo de llama que usé en el punto de acceso a datos estadísticos, pero ajustando el prompt a los nuevo requerimientos para ver qué tal funciona.

In [37]:
def clasificar_intencion_llm(pregunta_usuario):
    prompt = f"""<|system|>
Sos un asistente experto en el juego Meadow. Tu tarea es clasificar las consultas de los usuarios según la fuente de datos más adecuada para responderla.

Las posibles fuentes son:
- Información: para preguntas sobre reglas, estrategias, mecánica del juego, contenido textual general.
- Relaciones: para preguntas sobre personas que colaboraron, conexiones entre autores, ilustradores, editoriales, etc.
- Estadísticas: para preguntas sobre puntaje, cantidad de votos, rankings, dificultad, cantidad de jugadas, etc.

Formato de respuesta:
Información / Relaciones / Estadísticas

Ejemplos:

Usuario: ¿Cómo gano en el juego?
Respuesta: Información

Usuario: ¿Quién trabajó en la ilustración?
Respuesta: Relaciones

Usuario: ¿Qué puntaje obtuvo en el ranking?
Respuesta: Estadísticas

Usuario: ¿Cuántos fans tiene el juego Pradera?
Respuesta: Estadísticas

Usuario: ¿Qué mecánicas utiliza el juego Pradera?
Respuesta: Información

Usuario: ¿Quién diseñó el juego Pradera?
Respuesta: Relaciones

Usuario: ¿Cuándo fue lanzado el juego Pradera?
Respuesta: Relaciones

Usuario: {pregunta_usuario}
Respuesta:"""

    respuesta = llm(prompt, max_tokens=10, stop=["Usuario:"])
    return respuesta["choices"][0]["text"].strip()

Vamos a hacerle las mismas preguntas que le hicimos al otro modelo

In [38]:
clasificar_intencion_llm('quien creo el juego')

'Relaciones'

In [102]:
clasificar_intencion_llm('cuantos usuarios juegan')

'Estadísticas'

In [39]:
clasificar_intencion_llm('se puede jugar por equipos?')

'Información'

In [81]:
clasificar_intencion_llm('chi e il disegnatore grafico')

'Relaciones'

In [40]:
clasificar_intencion_llm('comment jouer')

'Información'

Buena respuesta en esta última query, que es la que el modelo NN no pudo predecir. Para realizar una buena comparación, vamos a hacer un pequeño set de preguntas para testear y calculamos los accuracy de ambos modelos

In [123]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Set de validación extendido
test_set = [
    ("¿Cuál es la puntuación promedio del juego?", "estadísticas"),
    ("¿Cómo se juega Pradera?", "información"),
    ("quién diseñó el juego?", "relaciones"),
    ("Cuántas personas tienen el juego en su colección?", "estadísticas"),
    ("cuales son las mecánicas del juego?", "información"),
    ("¿Quién ilustró el juego?", "relaciones"),
    ("¿Qué dificultad tiene el juego?", "estadísticas"),
    ("¿Cuántos votos recibió?", "estadísticas"),
    ("cuantas personas pueden jugar a la vez", "información"),
    ("en que año se lanzo pradera", "relaciones"),
    ("¿Quién escribió las reglas?", "relaciones"),
    ("que ranking ocupa", "estadísticas"),
    ("Cuántos comentarios tiene?", "estadísticas"),
    ("¿cómo funciona la mecánica de colocación de trabajadores?", "información"),
    ("¿qué diseñadores colaboraron entre sí?", "relaciones"),
    ("Qué editorial lo publicó?", "relaciones"),
    ("¿qué tipo de estrategia se puede usar?", "información"),
    ("Cuántos fans tiene el juego?", "estadísticas"),
    ("¿Qué recursos incluye el manual?", "información"),
    ("¿Quién editó la versión original?", "relaciones")
]

# Separar inputs y etiquetas reales
preguntas = [x[0] for x in test_set]
etiquetas_reales = [x[1] for x in test_set]

# Realizamos predicciones en la red neuronal
predicciones_nn = predecir_categoria_nn(preguntas, False)

# Métricas red neuronal
print("Accuracy red neuronal:", accuracy_score(etiquetas_reales, predicciones_nn))

# Evaluar LLM sobre el mismo set
def evaluar_llm_clasificador(preguntas, etiquetas_reales):
    pred_llm = [clasificar_intencion_llm(p).strip().lower() for p in preguntas]
    etiquetas_reales = [e.lower() for e in etiquetas_reales]
    acc = accuracy_score(etiquetas_reales, pred_llm)
    return acc
#Métricas LLM
accuracy_llm = evaluar_llm_clasificador(preguntas, etiquetas_reales)
print("LLM Accuracy:", accuracy_llm)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Accuracy red neuronal: 0.7
LLM Accuracy: 0.95


Con estas métricas se puede comparar y ver claramente que el clasificador basado en un LLM es superior a la red neuronal que hicimos anteriormente. Con un buen prompt se logró que realice gran cantidad de clasificaciones correctas, aunque la red neuronal también logra un accuracy bastante alto. Este último modelo podría mejorarse ampliando su dataset de entrenamiento, pero con las métricas obtenidas se nota que para la tarea de este punto el modelo basado en LLM es mejor, más eficaz y práctico.

## Pipeline de Recuperación (Retrieval)

### Para consultas en BBDD semánticas

####Búsqueda híbrida: semántica + por palabras clave

In [41]:
!pip install rank_bm25

In [42]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.schema import Document
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')

# 1. BM25 Retriever con mis fragmentos ya guardados en 'documentos_fragmentados'
def tokenize(doc):
    return word_tokenize(doc.page_content.lower())

bm25_retriever = BM25Retriever.from_documents(documentos_fragmentados)
bm25_retriever.k = 5  #podemos ajustar

# 2. Wrapper para mi búsqueda semántica existente (busqueda_semantica definida como interfaz de BBDD vectorial)
class CustomSemanticRetriever:
    def __init__(self, k=5):
        self.k = k

    def get_relevant_documents(self, query):
        resultados = busqueda_semantica(query, k=self.k)
        return [Document(page_content=r.page_content, metadata=r.metadata) for r in resultados]

semantic_retriever = CustomSemanticRetriever(k=5)

# 3. Retriever Híbrido (BM25 + Semántica)
def retriever_hibrido(query, k_total=5):
    """
    Búsqueda híbrida: mezcla BM25 y búsqueda semántica.
    Combina resultados, priorizando relevancia cruzada y eliminando duplicados.
    """
    # Recupera resultados
    resultados_bm25 = bm25_retriever.invoke(query)
    resultados_semanticos = busqueda_semantica(query, k=k_total)

    # Combina ambos (elimina duplicados por contenido)
    texto_visto = set()
    resultados_combinados = []
    for doc in resultados_bm25 + resultados_semanticos:
        if doc.page_content not in texto_visto:
            resultados_combinados.append(doc)
            texto_visto.add(doc.page_content)

        if len(resultados_combinados) >= k_total:
            break

    return resultados_combinados

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Ejemplo de uso:

In [57]:
query = "como gano el juego?"
resultados = retriever_hibrido(query, 4)

for i, r in enumerate(resultados):
    print(f"\nResultado {i+1}:\n{r.page_content[:]}")


Resultado 1:
Gracias como siempre compañero. 
2 A mi me tiene encandilado. No deja de ser un juego de draft y colecciones, pero el sistema matricial del quadropolis eleva el nivel de exigencia y la competición por las bonificaciones el nivel de interacción.

Resultado 2:
Avanzarán guiados por la pasión, 

la curiosidad por el mundo, una mente inquisitiva y el deseo de descubrir los misterios de la naturaleza y conver-
tirse en el observador más habilidoso. Ganará quien consiga más puntos observando los diferentes tipos de ani-
males, plantas y paisajes, así como reuniendo recuerdos durante su viaje. La competición continúa en la hoguera, 
donde los jugadores compiten por cumplir con los objetivos de sus aventuras.

Resultado 3:
Y es que Pradera propone a los jugadores tomar el papel de exploradores que registrarán los hallazgos que logren visualizar durante su travesía, como pueden ser bellos ejemplares, tanto de flora como de fauna, diversos tipos de terreno, paisajes embriagadores y

#### Re-ranking

In [43]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer

#Uso el mismo modelo multilingüe que ya me funcionó
rerank_model = SentenceTransformer("sentence-transformers/distiluse-base-multilingual-cased-v1")

def rerank_resultados(query, documentos, model, top_k=5):
    """
    Reordena los documentos según su similitud semántica con la consulta.
    """
    # Embedding del query
    embedding_query = model.encode([query])

    # Embeddings de los documentos
    textos = [doc.page_content for doc in documentos]
    embeddings_docs = model.encode(textos)

    # Similaridades
    similitudes = cosine_similarity(embedding_query, embeddings_docs)[0]

    # Ordenamos según similitud
    indices_ordenados = np.argsort(similitudes)[::-1]
    documentos_ordenados = [documentos[i] for i in indices_ordenados[:top_k]]

    return documentos_ordenados

def buscar_fragmentos_hibrido_rerank(query, k=5):
    # 1. Buscar documentos con método híbrido
    resultados = retriever_hibrido(query, k_total=k*2)  # recuperamos más para rerankear

    # 2. Aplicar rerank
    resultados_rankeados = rerank_resultados(query, resultados, model=rerank_model, top_k=k)

    return resultados_rankeados

In [56]:
resultados_finales = buscar_fragmentos_hibrido_rerank("como gano el juego?",4)
for i, doc in enumerate(resultados_finales, 1):
    traduccion = traducir_a_espanol(doc.page_content)
    print(f"\nResultado {i}:\n{traduccion}\nFuente: {doc.metadata['fuente']}")


Resultado 1:
Puedes ganar Meadow si puedes mostrar los puntos más de victoria después de un número determinado de rondas. 

¿Cómo funciona? 

Antes del primer juego, debes armar algunos soportes para tarjetas de cartón.
Fuente: foro_reviews.txt

Resultado 2:
La mayoría de los puntos gana. Es algo que es como un gastador. 

Realmente me gusta este juego mientras juego, pero siempre extraño los juegos que no tienen la tensión de un objetivo.
Fuente: foro_variantes.txt

Resultado 3:
No puedo llegar a exposiciones, etc. Entonces, ¿cómo alguien como yo tiene sus manos en las promociones? 
Boantziggy
@Boardziggy
Tengo muchas ganas de obtener este juego, pero soy un complemento. Aquí en el Reino Unido es costoso obtener las promociones y, no sé, me quedo molesto cuando hay más contenido, pero está fuera de su alcance.
Fuente: foro_general.txt

Resultado 4:
Siempre acabo las partidas con la sensacion de que me falta una ronda y por qué se acaba tan pronto… xD
3 Gran reseña, como siempre. Está

###Para consultas en BBDD de grafos y tabulares

En el apartado 'Interfaz' de 'Acceso a los Datos Estadísticos' creamos una función que, a partir de una query del usuario, genera un filtro de Pandas para la tabla (extraer_filtro_estadistico). También definimos busqueda_tabular que sistematiza todas las funciones realizadas allí: recibe una query y devuelve el resultado correspondiente extraído de la tabla. Mostramos un ejemplo de uso:

In [94]:
busqueda_tabular("que puntaje promedio tiene meadow")

,Estadistica,Valor
0,Avg. Rating,7.719


Lo mismo para la consulta a la base de grafos: fue realizada la función 'traducir_a_cypher' que recibe una consulta y la traduce en una consulta cypher. Se definió la función 'busqueda_grafo' para traer al usuario la respuesta a su consulta en lenguaje natural, utilizando el filtro generado por 'traducir_a_cypher', cumpliendo con el dinamismo pedido. Ejemplo de uso:

In [69]:
busqueda_grafo("quien es el ilustrador de pradera")

[{'n.nombre': 'Karolina Kijak'}, {'n.nombre': 'Katarzyna Fiebiger'}]

##Integración - Loop Conversacional

In [44]:
!pip install deep-translator langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 38.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=1a3d7d92ac998f3250c0a9522a926e043c332026d7b77c3d08fd85def49780d9
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect


In [45]:
from deep_translator import GoogleTranslator
from langdetect import detect
from llama_cpp import Llama
from datetime import datetime

In [55]:
# === Cargar modelo LLM local ===
#Usamos el mismo que implementamos hasta ahora
# model_path = hf_hub_download(
#     repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
#     filename="mistral-7b-instruct-v0.2.Q4_K_M.gguf"
# )
# llm = Llama(
#     model_path=model_path,  #← AJUSTAR
#     n_ctx=2048,
#     n_threads=4,
#     verbose=False
# )

# === Traductor ===
def detectar_idioma(texto):
    try:
        return detect(texto)
    except:
        return "es"

def traducir(texto, destino):
    try:
        return GoogleTranslator(source='auto', target=destino).translate(texto)
    except:
        return texto

# === Chatbot ===
def chatbot():
    memoria_conversacion = []
    print("👋💬 Bienvenido al chatbot experto en Pradera. Escribí tu consulta o 'salir' para terminar.")

    while True:
        user_input = input("Usuario: ")
        if user_input.lower() in ['salir', 'exit', 'quit','q']:
            print("🤓 ¡Hasta luego!")
            break

        idioma_usuario = detectar_idioma(user_input)
        categoria = clasificar_intencion_llm(user_input)

        # === Recuperación según categoría ===
        if categoria == "Información":
          contexto_docs = buscar_fragmentos_hibrido_rerank(user_input, k=3)

          # Filtramos fragmentos breves y sin opiniones personales
          contexto_filtrado = [
              traducir_a_espanol(doc.page_content)
              for doc in contexto_docs
              if len(doc.page_content) < 350 and 'me gusta' not in doc.page_content.lower()
          ]

          contexto_texto = "\n".join(contexto_filtrado)

        elif categoria == "Estadísticas":
            resultado = busqueda_tabular(user_input)
            contexto_texto = f"Resultado de la tabla: {resultado.to_string(index=False)}"

        elif categoria == "Relaciones":
            resultado = busqueda_grafo(user_input)
            contexto_texto = f"Resultado del grafo: {resultado}"

        else:
            contexto_texto = "No se encontró una categoría válida para la consulta."

        # === Prompt final ===
        prompt = f"""<|system|>
Sos un asistente experto en el juego de mesa Pradera. Tu tarea es responder con claridad, en el MISMO idioma del usuario, y de forma breve y concreta.

- NO repitas información.
- NO mezcles idiomas.
- Si no hay datos concretos en el contexto, decí que no sabés y sugerí reformular la pregunta.
- Evitá mencionar temas irrelevantes como frustraciones, precios, disponibilidad o emociones personales.
- Centrate exclusivamente en reglas, funcionamiento o datos objetivos del juego.

Historial de conversación:
{chr(10).join(memoria_conversacion[-5:])}

Consulta del usuario:
{user_input}

Información relevante:
{contexto_texto}

Respuesta:
"""

        salida = llm(prompt, max_tokens=256, stop=["Usuario:", "Tú:"])
        respuesta = salida["choices"][0]["text"].strip()

        if idioma_usuario != "es":
            respuesta = traducir(respuesta, destino=idioma_usuario)

        memoria_conversacion.append(f"Usuario: {user_input}")
        memoria_conversacion.append(f"Asistente: {respuesta}")

        print(f"Asistente: {respuesta}")

In [56]:
chatbot()

👋💬 Bienvenido al chatbot experto en Pradera. Escribí tu consulta o 'salir' para terminar.
Usuario: como gano el juego?
Asistente: El objetivo del juego de Meadow es mostrar mayor cantidad de puntos de victoria que tu oponente después de un número determinado de rondas. Los puntos se ganan principalmente mediante la colocación de tarjetas de colores en el tablero. Las reglas del juego son simples de aprender y el juego puede ser jugado en una variedad de niveles de complejidad. La primera persona en llegar a un número determinado de puntos de victoria gana el juego. Para prepararse para el juego, debes armar algunos soportes para las tarjetas de cartón antes de comenzar.
Usuario: que ranking ocupa el juego?
Asistente: El juego de Meadow ocupa el puesto 209 en la clasificación de Valor de la estadística.
Usuario: quien diseño el juego?
Asistente: El juego de Meadow fue diseñado por Klemens Kalicki.
Usuario: que estrategias de juego puedo usar?
Asistente: En el juego de Meadow, las estrat

In [57]:
chatbot()

👋💬 Bienvenido al chatbot experto en Pradera. Escribí tu consulta o 'salir' para terminar.
Usuario: how many can play this game?
Asistente: Up to six players can play Pradera.
Usuario: when was the game released?
Asistente: I don't have the specific release date for Pradera at hand. To find out, please check the game's official website or its rulebook.
Usuario: tell me the year of release. check the relations of the game
Asistente: I'm unable to give you the exact year of release for Pradera from this context. However, I can tell you that some data sources indicate that the game was released in the year 2021. To confirm, please refer to the game's official website or its rulebook.
Usuario: quit
🤓 ¡Hasta luego!


# Ejercicio 2 - Agente Autónomo

In [103]:
!pip install langchain openai

In [110]:
!pip install -U langchain langchain-community duckduckgo-search

In [ ]:
!pip install wikipedia

In [115]:
from langchain.tools import Tool
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.prompts import PromptTemplate
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper

In [219]:
# === Paso 1: Creación de herramientas (Tools) ===
def doc_search(query):
    """Busca y sintetiza texto relevante sobre Pradera."""
    resultados = buscar_fragmentos_hibrido_rerank(query, k=4)
    texto = "\n".join([traducir_a_espanol(doc.page_content) for doc in resultados])
    # (opcional) limpiar o resumir el contenido con un mini-LLM local o regex
    return texto[:1500]

def graph_search(query):
    """Responde consultas dinámicas relacionadas a relaciones entre entidades del juego usando Neo4j."""
    resultado = busqueda_grafo(query)
    if not resultado:
        return "No se encontró información."

    # Intentar detectar si es sobre diseñador
    if "diseñó" in query.lower() or "diseñador" in query.lower():
        try:
            nombre = resultado[0].get("p.nombre") or resultado[0].get("n.nombre")
            return [{"diseñador": nombre}]
        except:
            return resultado

    return resultado

def table_search(query):
    """Responde consultas dinámicas relacionadas a estadísticas del juego a partir de una tabla."""
    resultado = busqueda_tabular(query)

    if isinstance(resultado, str) and resultado == "NO_MATCH":
        return "NO_MATCH: No se encontró información relevante en la tabla."

    if resultado.empty:
        return "NO_MATCH: No se encontraron resultados."

    # Convertir a texto legible
    texto = "\n".join([
        f"{fila['Estadistica']}: {fila['Valor']}"
        for _, fila in resultado.iterrows()
    ])
    return texto

def wikipedia_search(query):
    """Para búsquedas en una enciclopedia libre."""
    wiki = WikipediaAPIWrapper(lang="es")
    return wiki.run(query)

def duckduckgo_search(query):
    """Para búsquedas en internet."""
    ddg = DuckDuckGoSearchAPIWrapper()
    return ddg.run(query)

# === Paso 2: Definir herramientas LangChain ===
tools = [
    Tool(
        name="buscador_documentos",
        func=doc_search,
        description="Consulta reglas, mecánicas, estrategias y contenido textual general de Pradera."
    ),
    Tool(
        name="buscador_tabla",
        func=table_search,
        description="Consulta métricas y estadísticas del juego, como puntuaciones o rankings."
    ),
    Tool(
        name="buscador_grafo",
        func=graph_search,
        description="Recupera relaciones entre autores, ilustradores, editoriales y demás entidades."
    ),
    Tool(
        name="wikipedia_search",
        func=wikipedia_search,
        description="Busca información enciclopédica en Wikipedia sobre Pradera o términos relacionados."
    ),
    Tool(
        name="duckduckgo_search",
        func=duckduckgo_search,
        description="Realiza búsquedas generales en la web cuando las fuentes internas no bastan."
    ),
]

# === Paso 3: Inicializar LLM y agente ===
llm_2 = ChatOpenAI(
    model_name="mistralai/Mistral-7B-Instruct-v0.1",
    openai_api_base="https://api.together.xyz/v1",
    openai_api_key="99ce5f5ea2eab89f2e85537d6fc74fb6d40388401de0789efe14a2758f4aa88d",
    temperature=0.2
)

agente = initialize_agent(
    tools=tools,
    llm=llm_2,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    agent_kwargs={
        "prefix": """Sos un experto en el juego de mesa Pradera. Tu objetivo es responder a las preguntas utilizando las herramientas disponibles.

IMPORTANTE: -PROHIBIDO usar la misma herramienta dos veces seguidas.
-Si pasaste por todas las herramientas y no hay nada relevante, RESPONDE: pedile al usuario que reformule la pregunta, que no encontras la informacion.

Tenés acceso a estas herramientas:
- buscador_documentos: busca reglas, estrategias y contenido textual.
- buscador_tabla: busca métricas y estadísticas como puntuación o rankings.
- buscador_grafo: busca relaciones entre entidades del juego.
- wikipedia_search: consulta enciclopedia libre sobre Pradera u otros temas.
- duckduckgo_search: búsqueda general en internet.

Reglas:
-Usá 'Thought', 'Action' y 'Action Input' para razonar y actuar paso a paso.
-Nunca escribas Action: herramienta(input).
-Siempre escribí Action y Action Input en líneas separadas.
-Pasa por TODAS LAS HERRAMIENTAS antes de dar una respuesta.
-Cuando uses 'buscador_documentos', extraé únicamente la información útil y concreta. Si el resultado contiene texto informal, comentarios irrelevantes o fragmentos confusos, ignoralos.
-Si ya obtuviste una respuesta clara y concreta, NO vuelvas a usar una herramienta.
-Si la respuesta incluye un dato específico (como "30-45 minutos" o "2-4 jugadores"), asumí que ya ES CONCRETA y ya podés responder.
-Si una herramienta devuelve "NO_MATCH", NO la vuelvas a usar para esa consulta. En ese caso, intentá usar otra herramienta diferente, o bien decí que no hay información disponible.
-SIEMPRE se CONCISO con las respuestas. SOLO brinda la información relevante.
-PROHIBIDO inventar. Solo responde con la informacion veridica que encuentres en los documentos.
-NO rellenes respuestas. NO repitas información.
-NO mezcles idiomas.
-Si no hay datos concretos en el contexto, decí que no sabés y sugerí reformular la pregunta.
-Usá el siguiente formato para razonar:

Thought: <tu razonamiento>
Action: <nombre de herramienta>
Action Input: <consulta>
Observation: <resultado de la herramienta>
... (más Thought/Action/Observation si hace falta)
Final Answer: <respuesta clara y en el mismo idioma que la pregunta>

Ejemplo:
Question: ¿Qué mecánicas tiene el juego Pradera?
Thought: Necesito consultar los documentos.
Action: buscador_documentos
Action Input: ¿Qué mecánicas tiene el juego Pradera?
Observation: El juego utiliza Hand Management, Drafting y Set Collection.
Thought: Con eso puedo dar la respuesta.
Final Answer: Las mecánicas principales de Pradera son Hand Management, Drafting y Set Collection.

"""
    }
)


# === Paso 4: Función para usar el agente ===
def agente_autonomo(pregunta):
    # Detectar idioma original del usuario
    idioma_usuario = detectar_idioma(pregunta)

    # Ejecutar agente con la pregunta original
    try:
        respuesta = agente.invoke({
            "input": pregunta,
            "chat_history": []  # si usás memoria real, reemplazá esto
        })
    except Exception as e:
        print("Error al ejecutar el agente:", e)
        return "Ocurrió un error al procesar la consulta."

    salida = respuesta["output"]

    idioma_salida = detectar_idioma(salida)

    if idioma_salida != idioma_usuario:
        salida = traducir(salida, destino=idioma_usuario)

    print("\n🤖 Respuesta:", salida)
    return salida

In [210]:
agente_autonomo("cuales son las mecanicas del juego?")



> Entering new AgentExecutor chain...
 Thought: Do I need to use a tool? Yes
Action: buscador_documentos
Action Input: ¿Qué mecánicas tiene el juego Pradera?
Observation: 5 Hola Iván, la verdad es que me has sorprendido con el sobresaliente, yo esperaba un notable al no implementar mecánicas especialmente novedosas. Desde luego el juego es bonito a más no poder. Estaba pensando comprarme el It’s a wonderful world, cuando ha salido este y me ha hecho dudar, porque a mi modo de ver aunque no lo parezca, tienen mecánicas similares.
Y Pradera es de esos juegos que entran solos en cualquier ludoteca porque lo tiene todo como juego cuasi familiar. 
20 Saludos excelente revisión, existe un juego pequeño que se llama  Codex Naturalis, y dicen q es muy parecido a Pradera, tanto asi que le llaman minipradera. Será cierto eso?
¿Qué contiene? ¿Alguien sabe? Tengo casi todas las promociones para este juego y la expansión posterior, pero no esta. Es la primera vez que escuché sobre eso y tengo muc

'Pradera es un juego de cartas con mecánica de borrador y recolección a la que es accesible conceptualmente y requiere la optimización del juego mediante el uso de un sistema de selección de matriz y eligiendo qué tarjeta numerada usar en cada turno. Es ágil, escala bien, es moderadamente desafiante (pero adecuado para todas las audiencias) y tiene un aspecto visual excelente.'

In [209]:
agente_autonomo("en que se basa el juego?")



> Entering new AgentExecutor chain...
 Thought: Necesito consultar los documentos.
Action: buscador_documentos
Action Input: ¿Qué se basa el juego Pradera?
Observation: ¿Qué contiene? ¿Alguien sabe? Tengo casi todas las promociones para este juego y la expansión posterior, pero no esta. Es la primera vez que escuché sobre eso y tengo mucha curiosidad.
Y Pradera es de esos juegos que entran solos en cualquier ludoteca porque lo tiene todo como juego cuasi familiar. 
20 Saludos excelente revisión, existe un juego pequeño que se llama  Codex Naturalis, y dicen q es muy parecido a Pradera, tanto asi que le llaman minipradera. Será cierto eso?
Napas y caos del juego
@Gamenapsandchaos
Me escribieron ahora pero no tuvieron suerte ...
"Hola,
Gracias por el mensaje. Póngase en contacto con la tienda donde se realizó la compra o el distribuidor de juegos en su país. "¿Cómo puedo saber qué distribuidor está en Rumania para Meadow?
10 Gran reseña iMisut (como todas). 
Recomiendas este juego para

'Pradera es un juego de cartas que se basa en el concepto de "gestión de manos", "redacción" y "colección establecida". Es un juego que se puede jugar solo en cualquier tienda de juegos porque tiene todos los elementos de un juego familiar. Hay un pequeño juego llamado Codex Naturalis que a menudo se conoce como "Minipradera" porque es similar a Pradera. Es posible que Codex Naturalis sea un juego similar a Pradera, pero no está claro si esto es cierto. Si tiene alguna pregunta sobre el juego, puede comunicarse con la tienda donde la compró o al distribuidor de juegos en su país.'

In [160]:
agente_autonomo('el juego es para mayores de cuanto?')



> Entering new AgentExecutor chain...
 Thought: Do I need to use a tool? No
AI: El juego Pradera es para mayores de 12 años.

> Finished chain.

🤖 Respuesta: El juego Pradera es para mayores de 12 años.


'El juego Pradera es para mayores de 12 años.'

In [194]:
agente_autonomo('which familiar ranking does it occupies')



> Entering new AgentExecutor chain...
 Thought: I need to use a tool to find the ranking of the familiar.
Action: buscador_tabla
Action Input: ranking familiar
Observation: Family Rank: 28
Thought: AI: The familiar occupies the 28th ranking.

> Finished chain.

🤖 Respuesta: The familiar occupies the 28th ranking.


'The familiar occupies the 28th ranking.'

In [199]:
agente_autonomo('how long does a game last on average')



> Entering new AgentExecutor chain...
 Thought: Do I need to use a tool? Yes
Action: buscador_tabla
Action Input: average game length of Pradera
Observation: NO_MATCH: No se encontraron resultados.
Thought: Thought: Do I need to use a tool? Yes
Action: buscador_tabla
Action Input: average game length of Pradera
Observation: NO_MATCH: No se encontraron resultados.
Thought: Thought: Do I need to use a tool? Yes
Action: buscador_tabla
Action Input: average game length of Pradera
Observation: NO_MATCH: No se encontraron resultados.
Thought: Thought: Do I need to use a tool? Yes
Action: buscador_tabla
Action Input: average game length of Pradera
Observation: NO_MATCH: No se encontraron resultados.
Thought: Thought: Do I need to use a tool? No
AI: The average game length of Pradera is 30-45 minutes.

> Finished chain.

🤖 Respuesta: The average game length of Pradera is 30-45 minutes.


'The average game length of Pradera is 30-45 minutes.'

In [217]:
agente_autonomo("se pueden formar equipos jugando?")



> Entering new AgentExecutor chain...
 Thought: Do I need to use a tool? Yes
Action: buscador_documentos
Action Input: ¿Se pueden formar equipos jugando Pradera?
Observation: ¿Sería posible jugar solo la mitad del juego (es decir, nunca introducir las cartas N) y tener una experiencia similar? 
Booch
@Booch77
¿Sería posible jugar solo la mitad del juego (es decir?
¡Ya podemos comenzar! 
Una partida de Pradera se desarrolla a lo largo de un determinado número de rondas (6 en partidas a 2/3 jugadores, 8 a 4 jugadores). 
En cada ronda, comenzando por el jugador inicial y continuando en el sentido de las agujas del reloj, los jugadores alternarán turnos de acción (en partidas a 2 o 3 jugadores son 5 turnos por jugador, mientras que a 4 jugadores son 4 turnos por jugador).
Quiero poder jugar con 4 jugadores y la expansión aguas abajo. 
Pero lo intenté de esa manera el otro día y tomó 5 horas o más. 

¿Hay algunos elementos meta-juego que pueda introducir para reducir el tiempo?
Tengo wing

'Es posible jugar Pradera con 4 jugadores y la expansión de Aguas Abajo. Sin embargo, puede llevar más tiempo que un juego con menos jugadores. Una forma de reducir el tiempo es jugar con un mazo más corto, lo que reduciría la cantidad de cartas para elegir durante el draft. Además, podría considerar jugar con una variante del juego que tiene un tiempo de juego más corto, como Pradera: The Lost City.\n\nCon respecto a su pregunta sobre Wingspan y Pradera, son dos juegos diferentes con diferentes mecánicas y temas. Si bien pueden no superarse directamente, potencialmente podrían jugar juntos en la misma Ludoteca. Pradera podría proporcionar una capa adicional de estrategia y complejidad a su experiencia de juego, mientras que Wingspan podría ofrecer un tipo diferente de desafío y disfrute.'

In [220]:
agente_autonomo("dove puoi comprare")



> Entering new AgentExecutor chain...
 Thought: Do I need to use a tool? Yes
Action: duckduckgo_search
Action Input: "Where can I buy Pradera?"
Observation: Propagation 🌱 Seed Propagation Starting cucumber 'Pradera' from seeds is a rewarding process. Begin by sowing seeds indoors 4-6 weeks before the last frost, or you can directly plant them outdoors once the frost has passed. Germination typically takes 7-14 days under optimal conditions. Buying Online Drugs Safely Overview You can safely buy medicine online if you use online pharmacies recommended by the National Association of Boards of Pharmacy. This organization verifies Internet drugstores throughout the United States and most Canadian provinces. Can I change or cancel my order with Pradera after placing it? Please contact Pradera's customer service team as soon as possible if you need to change or cancel your order. Pradera processes orders quickly, but the team will do their best to accommodate your request if the order hasn

"Puoi acquistare Pradera online da varie farmacie online che sono raccomandate dalla National Association of Boards of Pharmacy. Se è necessario modificare o annullare l'ordine, contattare il team di servizio clienti di Pradera il prima possibile."